# Barter: Instruments & Market Data (Advanced)

This notebook demonstrates the full data layer using the Python bindings:
instruments, indexed lookups, trade streaming, L1/L2 order books, and
multi-exchange composition.

> This is the Python equivalent of the Rust `evcxr` notebook,
> using the same underlying Rust engine via PyO3 bindings.

## Topics Covered
- Core types: `ExchangeId`, `Instrument`, `IndexedInstruments`
- O(1) indexed lookups and JSON serialization
- Streaming public trades (12 exchanges)
- L1 order book streaming (best bid/ask)
- L2 order book manager with managed snapshots
- Multi-exchange, multi-kind unified streams
- Low-level integration architecture (Rust-only, explained)


## Prerequisites

```bash
cd /path/to/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```


In [1]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


Using barter from /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python/python


In [ ]:
import json
import time
from collections import Counter

from barter import (
    ExchangeId, Instrument, IndexedInstruments,
    Subscription, build_market_stream,
    build_order_book_manager,
)


---
## Part 1: Instruments & Index Management


### Core Types

Barter uses strongly-typed wrappers for all financial primitives:

- **`ExchangeId`** — identifies an exchange (e.g., `BINANCE_SPOT`, `OKX`)
- **`Instrument`** — a specific trading pair on an exchange
- **`IndexedInstruments`** — compact integer indices for O(1) lookups
- **`InstrumentKind`** — Spot, Perpetual, Future, or Option


In [3]:
# Define instruments across multiple exchanges
btc_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "btc_usdt", "BTCUSDT", "btc", "usdt")
eth_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "eth_usdt", "ETHUSDT", "eth", "usdt")
btc_okx = Instrument.spot(ExchangeId.OKX, "btc_usdt", "BTC-USDT", "btc", "usdt")
btc_perp = Instrument.new_instrument(
    ExchangeId.BINANCE_FUTURES_USD, "btc_usdt_perp", "BTCUSDT",
    "btc", "usdt", kind="perpetual",
)

for inst in [btc_spot, eth_spot, btc_okx, btc_perp]:
    print(f"{inst}  (kind={inst.kind}, base={inst.base}, quote={inst.quote_asset})")


binance_spot:BTCUSDT (spot)  (kind=spot, base=btc, quote=usdt)
binance_spot:ETHUSDT (spot)  (kind=spot, base=eth, quote=usdt)
okx:BTC-USDT (spot)  (kind=spot, base=btc, quote=usdt)
binance_futures_usd:BTCUSDT (perpetual)  (kind=perpetual, base=btc, quote=usdt)


### Indexed Lookups

In [4]:
indexed = IndexedInstruments([btc_spot, eth_spot, btc_okx, btc_perp])

print(f"IndexedInstruments: {indexed}")
print(f"  Exchanges:   {indexed.num_exchanges}")
print(f"  Assets:      {indexed.num_assets}")
print(f"  Instruments: {len(indexed)}")

print("\nExchanges:")
for idx, name in indexed.exchanges():
    print(f"  ExchangeIndex({idx}): {name}")

print("\nInstruments:")
for idx, name_int, name_ex, exchange, kind in indexed.instruments():
    print(f"  InstrumentIndex({idx}): {exchange}:{name_ex} ({kind})")


IndexedInstruments: IndexedInstruments(exchanges=3, assets=7, instruments=4)
  Exchanges:   3
  Assets:      7
  Instruments: 4

Exchanges:
  ExchangeIndex(0): binance_futures_usd
  ExchangeIndex(1): binance_spot
  ExchangeIndex(2): okx

Instruments:
  InstrumentIndex(0): binance_futures_usd:BTCUSDT (perpetual)
  InstrumentIndex(1): binance_spot:BTCUSDT (spot)
  InstrumentIndex(2): binance_spot:ETHUSDT (spot)
  InstrumentIndex(3): okx:BTC-USDT (spot)


### JSON Serialization

In [5]:
# Round-trip serialization
payload = btc_perp.to_json()
print("JSON:")
print(payload)

roundtrip = Instrument.from_json(payload)
print(f"\nRound-trip: {roundtrip}")


JSON:
{
  "exchange": "binance_futures_usd",
  "name_internal": "btc_usdt_perp",
  "name_exchange": "BTCUSDT",
  "underlying": {
    "base": {
      "name_internal": "btc",
      "name_exchange": "btc"
    },
    "quote": {
      "name_internal": "usdt",
      "name_exchange": "usdt"
    }
  },
  "quote": "UnderlyingQuote",
  "kind": {
    "perpetual": {
      "contract_size": "1",
      "settlement_asset": {
        "name_internal": "usdt",
        "name_exchange": "usdt"
      }
    }
  },
  "spec": null
}

Round-trip: binance_futures_usd:BTCUSDT (perpetual)


### Architecture: Why Indexed?

| Approach | Lookup Cost | Cache Behaviour | Memory |
|----------|-------------|-----------------|--------|
| `dict[str, T]` | O(1) amortised, hash + compare | Cache-unfriendly | Higher (string allocs) |
| `list[T]` + `Index(int)` | O(1) guaranteed | Cache-friendly (contiguous) | Minimal |

The `IndexedInstruments` builder resolves all string identifiers to compact
indices at startup. All hot-path code uses integer indices for direct array access.


---
## Part 2: Market Data Streaming


### Streaming Public Trades

`build_market_stream()` creates WebSocket connections and yields a unified
async stream of `MarketEvent` values from any combination of exchanges.


In [6]:
async def collect_trades(subscriptions, limit=5):
    stream = build_market_stream(subscriptions)
    events = []
    async for event in stream:
        if event.trade:
            events.append({
                "exchange": str(event.exchange),
                "instrument": event.instrument,
                "side": str(event.trade.side),
                "price": event.trade.price,
                "quantity": event.trade.amount,
            })
        if len(events) >= limit:
            break
    return events

# Stream BTC trades from Binance Spot
trades = await collect_trades([Subscription("binance_spot", "btc", "usdt")], limit=5)
for t in trades:
    print(f"  [{t['exchange']}] {t['instrument']} {t['side']} {t['price']} x {t['quantity']}")


  [binance_spot] btc_usdt_spot sell 72911.48 x 0.00751
  [binance_spot] btc_usdt_spot sell 72911.48 x 0.00027
  [binance_spot] btc_usdt_spot buy 72911.49 x 0.00017
  [binance_spot] btc_usdt_spot sell 72911.48 x 0.01196
  [binance_spot] btc_usdt_spot buy 72911.49 x 0.00136


### L1 Order Book (Best Bid/Ask)

In [7]:
async def collect_l1(subscriptions, limit=5):
    stream = build_market_stream(subscriptions)
    events = []
    async for event in stream:
        if event.kind == "order_book_l1":
            events.append({
                "exchange": str(event.exchange),
                "instrument": event.instrument,
                "best_bid": event.best_bid_price,
                "best_ask": event.best_ask_price,
            })
        if len(events) >= limit:
            break
    return events

l1 = await collect_l1([
    Subscription("binance_spot", "btc", "usdt", data_kind="order_book_l1"),
], limit=5)
for e in l1:
    print(f"  [{e['exchange']}] {e['instrument']} bid={e['best_bid']} ask={e['best_ask']}")


  [binance_spot] btc_usdt_spot bid=72911.48 ask=72911.49
  [binance_spot] btc_usdt_spot bid=72911.48 ask=72911.49
  [binance_spot] btc_usdt_spot bid=72911.48 ask=72911.49
  [binance_spot] btc_usdt_spot bid=72911.48 ask=72911.49
  [binance_spot] btc_usdt_spot bid=72911.48 ask=72911.49


### L2 Order Book Manager

`build_order_book_manager()` maintains local order books
with background incremental updates. Read snapshots at any time.


In [8]:
mgr = build_order_book_manager([
    Subscription("binance_spot", "btc", "usdt"),
])

time.sleep(3)  # wait for book to populate

book = mgr.get("btc_usdt_spot")
if book:
    print(f"BTC/USDT L2 OrderBook: {book}")
    print(f"  Best bid: {book.best_bid}")
    print(f"  Best ask: {book.best_ask}")
    print(f"  Spread:   {book.spread}")
    print(f"  Mid:      {book.mid_price}")
    print(f"  Top 3 bids: {book.bids[:3]}")
    print(f"  Top 3 asks: {book.asks[:3]}")
    snap = book.snapshot(5)
    print(f"  Depth-5 snapshot: {snap}")
else:
    print("Book not yet available")


BTC/USDT L2 OrderBook: OrderBook(bids=125, asks=125, mid=Some(72911.485), seq=91790940251)
  Best bid: (72911.48, 5.37837)
  Best ask: (72911.49, 2.13728)
  Spread:   0.010000000009313226
  Mid:      72911.485
  Top 3 bids: [(72911.48, 5.37837), (72911.47, 0.00072), (72911.1, 0.00014)]
  Top 3 asks: [(72911.49, 2.13728), (72911.5, 0.03354), (72911.99, 0.00016)]
  Depth-5 snapshot: {'bids': [(72911.48, 5.37837), (72911.47, 0.00072), (72911.1, 0.00014), (72910.53, 0.00027), (72910.52, 7e-05)], 'asks': [(72911.49, 2.13728), (72911.5, 0.03354), (72911.99, 0.00016), (72912.0, 0.00993), (72912.36, 7e-05)], 'mid_price': 72911.485, 'spread': 0.010000000009313226, 'sequence': 91790940251}


---
## Part 3: Multi-Exchange Streams


In [9]:
async def collect_multi(subscriptions, limit=10):
    stream = build_market_stream(subscriptions)
    events = []
    async for event in stream:
        row = {"exchange": str(event.exchange), "instrument": event.instrument, "kind": event.kind}
        if event.trade:
            row["price"] = event.trade.price
        elif event.kind == "order_book_l1":
            row["bid"] = event.best_bid_price
            row["ask"] = event.best_ask_price
        events.append(row)
        if len(events) >= limit:
            break
    return events

# Mix trades and L1 from multiple exchanges
multi = await collect_multi([
    Subscription("binance_spot", "btc", "usdt", data_kind="trades"),
    Subscription("binance_spot", "btc", "usdt", data_kind="order_book_l1"),
    Subscription("coinbase", "btc", "usd", data_kind="trades"),
], limit=10)

for e in multi:
    print(f"  [{e['kind']:>15}] {e['exchange']:>15} {e['instrument']}")

print(f"\nBy exchange: {Counter(e['exchange'] for e in multi)}")
print(f"By kind:     {Counter(e['kind'] for e in multi)}")


  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [          trade]        coinbase btc_usd_spot
  [  order_book_l1]    binance_spot btc_usdt_spot
  [  order_book_l1]    binance_spot btc_usdt_spot
  [  order_book_l1]    binance_spot btc_usdt_spot

By exchange: Counter({'coinbase': 7, 'binance_spot': 3})
By kind:     Counter({'trade': 7, 'order_book_l1': 3})


---
## Supported Exchanges

| Exchange | Python Name | Trades | L1 | L2 |
|----------|-------------|--------|----|----|  
| Binance Spot | `binance_spot` | Yes | Yes | Yes |
| Binance Futures | `binance_futures_usd` | Yes | Yes | Yes |
| OKX | `okx` | Yes | — | — |
| Coinbase | `coinbase` | Yes | — | — |
| Kraken | `kraken` | Yes | Yes | — |
| Bitfinex | `bitfinex` | Yes | — | — |
| Bitmex | `bitmex` | Yes | — | — |
| Bybit Spot | `bybit_spot` | Yes | Yes | Yes |
| Bybit Perpetuals | `bybit_perpetuals_usd` | Yes | Yes | Yes |
| Gateio Spot | `gateio_spot` | Yes | — | — |
| Gateio Perpetuals | `gateio_perpetuals_usd` | Yes | — | — |
| Gateio Futures | `gateio_futures_usd` | Yes | — | — |


---
## Low-Level Integration Architecture (Rust-Only)

The Python bindings use the high-level `build_market_stream()` and
`build_order_book_manager()` APIs. Under the hood, the Rust `barter-integration`
crate provides low-level traits for building custom exchange integrations:

```
WebSocket → Parser → Validator → Transformer → ExchangeStream
```

| Trait | Purpose |
|-------|---------|
| `Transformer` | Stateful data transformation (raw JSON → normalised events) |
| `Validator` | Input validation (sequence numbers, message integrity) |
| `Unrecoverable` | Classify errors as fatal vs transient |
| `Terminal` | Mark events that end a stream's lifecycle |

These traits are Rust-only and used internally by `barter-data` for all
12 exchange implementations. Python users consume the normalised output
via `MarketEvent` objects without needing to interact with these traits.
